In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss



In [2]:
train_data = pd.read_csv("/kaggle/input/lmsys-chatbot-arena/train.csv")
test_data = pd.read_csv("/kaggle/input/lmsys-chatbot-arena/test.csv")
sample_sub = pd.read_csv("/kaggle/input/lmsys-chatbot-arena/sample_submission.csv")



In [3]:
# Keep original text construction idea, but use a single combined input text consistently
train_data["text"] = (
    train_data["prompt"].fillna("")
    + " "
    + train_data["response_a"].fillna("")
    + " "
    + train_data["response_b"].fillna("")
)
test_data["text"] = (
    test_data["prompt"].fillna("")
    + " "
    + test_data["response_a"].fillna("")
    + " "
    + test_data["response_b"].fillna("")
)



In [4]:
# Convert the 3 one-hot target columns into a single multiclass label.
# This aligns with the competition metric (multiclass log loss) and avoids non-normalized independent probs.
target_cols = ["winner_model_a", "winner_model_b", "winner_tie"]
y_multiclass = train_data[target_cols].values.argmax(axis=1)

X = train_data["text"].values



In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y_multiclass, test_size=0.2, random_state=42, stratify=y_multiclass
)



In [6]:
vectorizer = CountVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)



In [7]:
# Single multinomial model outputs 3 probabilities that sum to 1 (required for correct multiclass log loss).
model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    multi_class="multinomial",
    solver="lbfgs",
)
model.fit(X_train_vec, y_train)



/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000, multi_class='multinomial', random_state=42)

In [8]:
# Proper multiclass validation log loss using 3-class probabilities
val_proba = model.predict_proba(X_val_vec)
val_loss = log_loss(y_val, val_proba, labels=[0, 1, 2])
print(f"Validation Log Loss (multiclass): {val_loss:.6f}")



Validation Log Loss (multiclass): 1.346974


In [9]:
# Predict on test with the same vectorizer/model
X_test_vec = vectorizer.transform(test_data["text"].values)
test_proba = model.predict_proba(X_test_vec)



In [10]:
# Build submission with exact required columns and correct id alignment.
# Using sample_submission ensures column order and row count match expected format.
submission_df = sample_sub[["id"]].copy()
submission_df["winner_model_a"] = test_proba[:, 0]
submission_df["winner_model_b"] = test_proba[:, 1]
submission_df["winner_tie"] = test_proba[:, 2]

# Ensure ids align with test_data ids; if sample_submission ids match test ids, this is already correct.
# If not, align explicitly by merging on id (safe and still minimal).
if not submission_df["id"].equals(test_data["id"]):
    tmp = pd.DataFrame(
        {
            "id": test_data["id"].values,
            "winner_model_a": test_proba[:, 0],
            "winner_model_b": test_proba[:, 1],
            "winner_tie": test_proba[:, 2],
        }
    )
    submission_df = sample_sub[["id"]].merge(tmp, on="id", how="left")

submission_df.to_csv("submission.csv", index=False)
print("Submission file created successfully: submission.csv")
print(submission_df.head())

Submission file created successfully: submission.csv
           id  winner_model_a  winner_model_b  winner_tie
0  3297560222        0.568663        0.163478    0.267858
1  2556155375        0.813645        0.121220    0.065134
2  1793939629        0.314200        0.361844    0.323956
3  2562993561        0.170078        0.511186    0.318736
4  2111345399        0.296465        0.123792    0.579743
